# Analyzing the final RLHF policy-suite evaluation

This notebook analyzes the completed 1024-token RLHF policy-suite run. It reads the consolidated suite artifacts instead of the old pairwise evaluation files.

Policies compared:

- `base`: Qwen2.5-0.5B-Instruct
- `sft_4096`: supervised fine-tuned LoRA policy
- `ppo_4096_ep2_u400`: PPO policy trained from SFT using the epoch-2 reward model

The evaluator generated all three responses once per prompt, scored them with the same reward model, and derived all pairwise comparisons from the same table.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# If this notebook is opened from notebooks/, move to repo root.
import os, sys
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EVAL_DIR = Path("outputs/rlhf/qwen25_05b_helpsteer3_eval_suite_4096_ep2_u400_eval1024")
SAMPLES_CSV = EVAL_DIR / "policy_suite_samples.csv"
SUMMARY_MD = EVAL_DIR / "policy_suite_summary.md"
PAIRWISE_CSV = EVAL_DIR / "policy_suite_pairwise_summary.csv"

print("Repo root:", REPO_ROOT)
print("Eval dir exists:", EVAL_DIR.exists())
print("Samples exists:", SAMPLES_CSV.exists())

In [ ]:
df = pd.read_csv(SAMPLES_CSV)
pairwise = pd.read_csv(PAIRWISE_CSV)

policies = ["base", "sft_4096", "ppo_4096_ep2_u400"]
reward_cols = [f"{p}_reward" for p in policies]

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head(3))

## Official summary generated by the evaluator

In [ ]:
display(Markdown(SUMMARY_MD.read_text()))

## Reward and length distributions

Absolute reward values are not calibrated. The useful quantities are comparisons **within the same prompt**.

In [ ]:
reward_summary = df[reward_cols].describe().T
reward_summary["skew"] = df[reward_cols].skew()
reward_summary["kurtosis"] = df[reward_cols].kurtosis()
display(reward_summary)

length_summary = pd.DataFrame({
    p: {
        "mean_tokens": df[f"{p}_response_tokens"].mean(),
        "median_tokens": df[f"{p}_response_tokens"].median(),
        "cap_hit_rate": df[f"{p}_cap_hit"].mean(),
        "empty_rate": df[f"{p}_empty"].mean(),
        "eos_hit_rate": df[f"{p}_hit_eos"].mean(),
    }
    for p in policies
}).T
display(length_summary)

In [ ]:
for p in policies:
    plt.figure(figsize=(7, 4))
    plt.hist(df[f"{p}_reward"], bins=60)
    plt.title(f"Reward distribution: {p}")
    plt.xlabel("Reward-model score")
    plt.ylabel("Count")
    plt.show()

## Domain-wise results

In [ ]:
domain_winners = pd.crosstab(df["domain"], df["winner"], normalize="index").round(4)
display(domain_winners)

for col in ["winner_base_vs_sft_4096", "winner_base_vs_ppo_4096_ep2_u400", "winner_sft_4096_vs_ppo_4096_ep2_u400"]:
    print("\n", col)
    display(pd.crosstab(df["domain"], df[col], normalize="index").round(4))

## Language-wise results for common languages

In [ ]:
language_counts = df["language"].value_counts()
common_langs = language_counts[language_counts >= 20].index
lang_winners = pd.crosstab(df[df.language.isin(common_langs)]["language"], df[df.language.isin(common_langs)]["winner"], normalize="index").round(4)
display(lang_winners.sort_values("ppo_4096_ep2_u400", ascending=False))

## Strong wins and strong losses

These tables are starting points for manual curation. Do **not** publish examples based only on reward-model deltas; inspect the full text first.

In [ ]:
strong_ppo_wins = df[
    (df["winner"] == "ppo_4096_ep2_u400") &
    (df["delta_ppo_4096_ep2_u400_minus_base"] > 2.0) &
    (df["delta_ppo_4096_ep2_u400_minus_sft_4096"] > 1.0)
].sort_values("delta_ppo_4096_ep2_u400_minus_base", ascending=False)

strong_ppo_losses = df[
    (df["winner_base_vs_ppo_4096_ep2_u400"] == "base") &
    (df["delta_ppo_4096_ep2_u400_minus_base"] < -5.0)
].sort_values("delta_ppo_4096_ep2_u400_minus_base")

cols = [
    "idx", "domain", "language", "winner", "reward_rank",
    "base_reward", "sft_4096_reward", "ppo_4096_ep2_u400_reward",
    "delta_ppo_4096_ep2_u400_minus_base",
    "delta_ppo_4096_ep2_u400_minus_sft_4096",
    "base_response_tokens", "sft_4096_response_tokens", "ppo_4096_ep2_u400_response_tokens",
]

print("Strong PPO wins:", len(strong_ppo_wins))
display(strong_ppo_wins[cols].head(25))

print("Strong PPO losses:", len(strong_ppo_losses))
display(strong_ppo_losses[cols].head(25))

## Human-readable response viewer

In [ ]:
def extract_last_user_prompt(rendered_prompt: str) -> str:
    matches = re.findall(r"<\|im_start\|>user\n(.*?)<\|im_end\|>", str(rendered_prompt), flags=re.S)
    if matches:
        return matches[-1].strip()
    return str(rendered_prompt).replace("<|im_start|>assistant\n", "").strip()


def show_example(idx: int, max_chars: int | None = None):
    row = df[df["idx"] == idx]
    if row.empty:
        raise ValueError(f"idx {idx} not found")
    r = row.iloc[0]
    display(Markdown(f"# idx {idx} — {r['domain']} / {r['language']} — winner: `{r['winner']}`"))
    display(Markdown("## Prompt\n" + extract_last_user_prompt(r["prompt"])))
    for p in policies:
        text = str(r[f"{p}_response"])
        if max_chars is not None and len(text) > max_chars:
            text = text[:max_chars] + "\n\n...[truncated for notebook display]"
        display(Markdown(
            f"## {p}\n"
            f"reward: `{r[f'{p}_reward']:.4f}` | tokens: `{int(r[f'{p}_response_tokens'])}` | "
            f"cap_hit: `{bool(r[f'{p}_cap_hit'])}`\n\n" + text
        ))

# Try a known example:
show_example(0, max_chars=2500)

## Export curation candidates

In [ ]:
def export_candidate_markdown(indices, path=EVAL_DIR / "curation_candidates.md", max_chars=2200):
    lines = ["# RLHF curation candidates", ""]
    for idx in indices:
        row = df[df["idx"] == idx]
        if row.empty:
            continue
        r = row.iloc[0]
        lines.append(f"## idx {idx} — {r['domain']} / {r['language']} — winner: `{r['winner']}`")
        lines.append("")
        lines.append("**Prompt**")
        lines.append("")
        lines.append(extract_last_user_prompt(r["prompt"])[:max_chars])
        lines.append("")
        for p in policies:
            text = str(r[f"{p}_response"])
            if len(text) > max_chars:
                text = text[:max_chars] + "\n\n...[truncated for curation preview]"
            lines.append(f"### {p} — reward `{r[f'{p}_reward']:.4f}`, tokens `{int(r[f'{p}_response_tokens'])}`, cap_hit `{bool(r[f'{p}_cap_hit'])}`")
            lines.append("")
            lines.append(text)
            lines.append("")
        lines.append("---")
        lines.append("")
    Path(path).write_text("\n".join(lines), encoding="utf-8")
    return Path(path)

candidate_indices = [2, 0, 1210, 1158, 418, 1511, 1970, 1303, 176, 1835, 1548, 346, 1281, 1579, 656, 86]
path = export_candidate_markdown(candidate_indices)
print("Wrote", path)